# Lab 1: ingest five formats into one indexed corpus

**Time: about 75 minutes. Work in pairs.** One person types, the other checks.

**What you will build.** You take seven real files in five formats, check what the loaders actually
pulled out of each one, cut that text into small pieces, attach labels to every piece, put the pieces
into a small search index, and ask it five real support questions. Two of the answers come back wrong,
and by the end you will know exactly why.

**How the cells work.** Each step has up to three cells. The first shows you the real thing. The
second is marked `YOUR CALL` and is the only one you edit, where you type your prediction or your
setting. The third measures what really happens and prints your answer next to it.

**Before you start.** Every `YOUR CALL` cell already holds a working value, so you can run the whole
notebook once from top to bottom to see the shape of it. Then come back to Step 1 and put in your own
answers. Do not skip them. The gap between your guess and the measurement is the part you remember.

**Two words used throughout.** A *chunk* is one piece of text after a document has been cut up.
*Metadata* is the set of labels carried by a chunk, such as the file it came from and the date the
document took effect.

In [ ]:
# Setup. This cell is the same in every lab notebook and has to run first.
import sys, pathlib, textwrap

# ROOT must point at the lab folder. You may have opened this notebook from the lab
# folder or from notebooks/, so check which, and step up one level if needed.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# src/ holds the shared lab code. Putting it on the search path is what makes
# `from freshcart import ...` work in the cells below.
sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

## Step 1. Look at the files before you trust them

**What you will do.** Run the next cell to list the seven files in `data/raw` with their type and
size. Then open the `YOUR CALL` cell and name the file you think comes out completely empty, and the
file whose format you think loses the most information even when the loader reports success. The cell
after that loads all seven and prints what each one really gave up.

**What you should see.** Seven files: four PDFs, one Word file, one PowerPoint file and one Excel
file, from 2.5 KB to 210 KB. The reveal prints seven rows with a character count each, and exactly one
row reads `EMPTY, needs OCR` with 0 characters, for `support_escalation_sop_scanned.pdf`. Nothing
raises an error.

**Why this step is here.** That silent empty file is the most common production failure in this whole
course. It is a scan, so it is a picture of text, and an ordinary PDF reader returns nothing from it.
If you do not catch it now, the question it should answer comes back wrong in Step 5, and in a real
system nobody notices for weeks.

In [ ]:
# List the files with their type and size. Nothing is opened or read yet, so a file
# that is only a picture of text still looks perfectly normal in this listing.
for p in sorted(RAW.glob("*")):
    print(f"{p.name:44s} {p.suffix:6s} {p.stat().st_size/1024:8.1f} KB")

In [ ]:
# YOUR CALL: change these two lines before you run the reveal
PREDICT_EMPTY = "support_escalation_sop_scanned.pdf"
PREDICT_WORST = "support_sla_matrix.docx"

In [ ]:
# load_all() picks the right reader for each format: PDF, Word, PowerPoint, Excel.
# verify() then counts what actually came out of each file. That second step is the one
# most pipelines skip, and it is the only reason the empty file below gets noticed.
from freshcart import load_all, verify
docs = load_all()      # one "document unit" per PDF page, per slide, per spreadsheet row,
                       # and one for a whole Word file: a unit differs by format
rows = verify(docs)    # per file: how many units, how many characters, and a status
print(f"{'source':44s} {'units':>5} {'chars':>7}  status")
for r in rows:
    print(f"{r['source']:44s} {r['units']:5d} {r['chars']:7d}  {r['status']}")

empty = [r["source"] for r in rows if r["chars"] == 0]
print("\nempty after extraction:", empty)
print("you predicted empty:", PREDICT_EMPTY, "->", "correct" if PREDICT_EMPTY in empty else "not this one")
print("nothing raised an exception. That is the point: the failure is silent.")

## Step 2. Read what the loader returned, not what you hoped for

**What you will do.** This step has three parts.

1. Run the next cell. It takes three files that all loaded without an error, a two-column contract
   PDF, a Word file with a table, and a slide deck, and shows what each one lost on the way in. Read
   the output with your pair. Then set `MIN_CHARS_PER_UNIT` in the `YOUR CALL` cell, which says how
   few characters per page or per slide means "this did not work". The next cell applies your rule
   to all seven files. The cell after it repairs the two losses that no rule can catch, and checks
   that each repair worked.
2. Run the same seven files through LangChain and LlamaIndex, and compare them with your loader.
3. Run two tools that read the layout for you, one free and one paid, and score every reader in
   this step on the same three questions.

**What you should see.** The contract PDF comes back one printed line at a time, so lines end in the
middle of sentences. Its two columns still come back in the right order, clauses 4.1 to 4.5. In the
Word file the heading "Turnaround targets by priority" is followed by "Notes", and the four table
rows sit at the very end, after the notes. Each row still carries its column names, such as
"First response TAT: 30 minutes", because this loader writes them in; in part 2 you will see framework
loaders that do not. On slide 2 of the deck, the text boxes were created bottom to top, so a loader
that reads them in creation order puts the title last; this loader sorts them top to bottom. With the
default rule of 50 characters, one file is held back: `support_escalation_sop_scanned.pdf`, 0
characters in 1 unit. Set the number too low and the list comes back empty.

In the repair cell, the contract's 30 printed lines become 12: each clause heading on a line of its
own, then each paragraph whole. The Word file is read top to bottom, so its four rows sit under
"Turnaround targets by priority" again, and each row carries that heading the way it already carries
its column names. Before the repair, the piece that holds the P3 row does not say which table it came
from. After the repair, it does. The repair also glues the contract's two title lines into one: every
repair guesses at the layout, so read its output the way you read the loader's.

**Why this step is here.** Nobody reads 40,000 files by hand, so the check has to be a rule that runs
by itself. This is the first quality gate of the pipeline, and in Week 4 the same idea becomes an
automatic test that can block a release. A rule can only hold back a file that came back empty. A
file that came back full but in the wrong shape needs a repair in the loader, and a check that shows
the repair still works.

In [ ]:
# Three files that all loaded without an error. Each one lost something different.
import textwrap
from pptx import Presentation

def show(name):
    units = [d for d in docs if d.meta.get("source") == name]
    print("=" * 90)
    print(f"{name}  |  {len(units)} unit(s)  |  labels on the first: {units[0].meta}")
    return units

# 1. The contract PDF is laid out in two columns and comes back one printed line at a time.
pdf = show("partner_agreement_extract.pdf")
lines = pdf[0].text.splitlines()
for line in lines[2:7]:
    print("   |", line)
print("-> Each line stops where the narrow column ran out of width, not where the sentence ends.")
print("   A splitter that cuts on line breaks would cut these sentences in half.")
print("   Clause order as read:", [l.split()[0] for l in lines if l[:2] == "4." and l[2:3].isdigit()])
print("   The left column was read in full before the right one, so 4.1 to 4.5 stayed in order.\n")

# 2. The Word file: every paragraph in the order the loader returned them.
sla = show("support_sla_matrix.docx")
for i, para in enumerate(sla[0].text.split("\n\n")):
    print(f"   {i:2d}  {textwrap.shorten(para, 80)}")
print("-> The heading 'Turnaround targets by priority' is followed by 'Notes', not by its table.")
print("   The table rows were moved to the very end, after the notes.\n")

# 3. The slide deck: the same slide, in two reading orders.
qbr = show("support_qbr_q1_2024.pptx")
slide = Presentation(str(RAW / "support_qbr_q1_2024.pptx")).slides[1]
print("   slide 2, in the order its text boxes were created in PowerPoint:")
for sh in slide.shapes:
    print("      ", textwrap.shorten(sh.text_frame.text, 76))
print("   slide 2, as our loader returns it, sorted top to bottom:")
for line in qbr[1].text.splitlines():
    print("      ", textwrap.shorten(line, 76))
print("-> A loader that reads text boxes in creation order puts the title last.")

In [ ]:
# YOUR CALL: the quarantine rule your pipeline would enforce
MIN_CHARS_PER_UNIT = 50

In [ ]:
# Apply your rule from the cell above: hold back any file whose TOTAL characters fall
# below MIN_CHARS_PER_UNIT times its unit count. A unit is a PDF page, a slide, a
# spreadsheet row, or a whole Word file.
quarantine = [r for r in rows if r["chars"] < MIN_CHARS_PER_UNIT * max(1, r["units"])]
print("would be quarantined by your rule:")
for r in quarantine:
    print(f"   {r['source']:44s} {r['chars']:6d} chars in {r['units']} unit(s)")
if not quarantine:
    print("   nothing. Your threshold is too low to catch the scanned file.")
print("\nIn production this list goes to OCR, not to the index.")

In [ ]:
# Repair the two losses your rule cannot catch. Both files came back full, just in the wrong
# shape. Each repair uses layout the file already carries, and each one is checked, not trusted.
import re
from docx import Document as Docx
from docx.table import Table
from freshcart import chunk_recursive

CLAUSE = re.compile(r"\d+(\.\d+)+\s")   # a numbered clause, such as "4.2 Partner-borne events"
ENDS = re.compile(r"[.:;!?]$")               # a line that finishes a sentence

def unwrap(text):
    """Rejoin the lines a PDF broke at the column edge. A line break stays only where a
    sentence ended, or next to a numbered clause heading."""
    lines = []
    for line in text.splitlines():
        broken = lines and not ENDS.search(lines[-1])       # the line above stopped mid-sentence
        clause = CLAUSE.match(line) or (lines and CLAUSE.match(lines[-1]))
        if broken and not clause:
            lines[-1] += " " + line
        else:
            lines.append(line)
    return "\n".join(lines)

def docx_in_order(path):
    """Read the body once, top to bottom, so each table stays under its heading, and write
    that heading into every row, the same way the loader already writes in the column names."""
    parts, heading = [], ""
    for block in Docx(str(path)).iter_inner_content():
        if isinstance(block, Table):
            rows = [[c.text.strip() for c in r.cells] for r in block.rows]
            for r in rows[1:]:
                parts.append(f"[{heading}] " + "; ".join(f"{h}: {v}" for h, v in zip(rows[0], r) if v))
        elif block.text.strip():
            if block.style.name.startswith("Heading"):
                heading = block.text.strip()
            parts.append(block.text.strip())
    return "\n\n".join(parts)

# 1. The contract PDF: rejoin the lines the column edge broke.
fixed_pdf = unwrap(pdf[0].text)
print(f"partner_agreement_extract.pdf: {len(pdf[0].text.splitlines())} printed lines, "
      f"{len(fixed_pdf.splitlines())} after the repair")
for line in fixed_pdf.splitlines()[:5]:
    print("   |", textwrap.shorten(line, 96))
print("-> The text of clause 4.1 is one line now, so no splitter can cut it at the column edge.")
print("   The two title lines were glued together too. A repair guesses, so read what it did.\n")

# 2. The Word file: read it in order, and write the heading into every row.
fixed_sla = docx_in_order(RAW / "support_sla_matrix.docx")
for i, para in enumerate(fixed_sla.split("\n\n")):
    print(f"   {i:2d}  {textwrap.shorten(para, 80)}")
for label, text in (("before", sla[0].text), ("after ", fixed_sla)):
    piece = next(p for p in chunk_recursive(text) if "Priority: P3" in p)
    print(f"   {label} the repair, the piece holding the P3 row names its table: "
          f"{'Turnaround targets' in piece}")
print("-> Each row is back under its heading, and names it wherever the chunker cuts.\n")

print("The slide deck needed no repair here: the loader already sorts its text boxes by position.")
print("The scanned file needs OCR, not a repair. Two tools in part 3 of this step read it.")
print("The rest of Lab 1 keeps the loader's own output, so your numbers match the guide.")

### Part 2. The same seven files, through two frameworks

**What you will do.** Point LangChain and LlamaIndex at the same `data/raw` folder, and put their
answers next to the loader you have been reading in this step. Then decide, for four specific
situations, which one you would actually reach for.

**What you should see.** Four head-to-head results, and not one of them makes a single framework
the winner.

- *Every format, no extra packages.* LlamaIndex opens all seven files. LangChain opens five: the
  PowerPoint and the Excel file need the `unstructured` package, which it does not bring with it.
- *Speed.* LlamaIndex takes roughly a third of LangChain's time on the same files. Your seconds will
  differ from ours; the ratio will not.
- *A spreadsheet row you can cite on its own.* Neither of them. LlamaIndex reads the sheet into one
  block of 2,415 characters whose columns all read `Unnamed`, because it takes the title banner as
  the header row. Ours gives nine rows, each one citable by itself.
- *Who wrote the file and when.* LangChain, and only LangChain. It is the one that carries `author`,
  `creationdate` and `producer`. Here they read "anonymous" and "untitled", because these files were
  generated. On a real corpus that field is how a governance filter knows what it is looking at.

Then the one they all lose. All three return zero characters for the scanned PDF and not one of them
says so. LlamaIndex does have `extraction_errors` and `extraction_warnings` fields. It attaches them
to the PowerPoint, which worked, and not to the scanned PDF, which did not.

**Why this step is here.** Somebody will ask you to use a framework. The question is never whether
the framework is good. It is which of three jobs it is doing for you: finding a reader for every
format, keeping the structure that makes a chunk answerable on its own, and telling you when
extraction failed. Expect one, sometimes two, never all three.

**Two words used here.** A *loader* turns a file into text. *Reader* is LlamaIndex's word for the
same thing.

In [ ]:
# The frameworks are installed in the first cell of this notebook. If you are running on your
# own machine and this cell asks you to install something, run the line it prints and re-run.
missing = []
for label, mod in [("langchain-community", "langchain_community"),
                   ("docx2txt", "docx2txt"),
                   ("llama-index-core", "llama_index.core"),
                   ("llama-index-readers-file", "llama_index.readers.file")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(label)

if missing:
    print("install these first, then re-run this cell:")
    print("   pip install " + " ".join(missing))
else:
    print("both frameworks are ready")

In [ ]:
# run_all points each loader at the same folder and records what came back and how long it
# took. per_file lines the answers up one row per file. A dash means that loader never opened
# the file at all, which is a different failure from opening it and getting nothing.
from loaders_compare import run_all, per_file, find_text, metadata_keys
from labkit import table

res = run_all()
by_file = per_file(res)      # not `rows`: your quarantine cell above still reads that
print(table(by_file, list(by_file[0].keys()), width=16))

print()
for name, r in res.items():
    not_read = ", ".join(f"{n} ({why})" for n, why in r["skipped"]) or "nothing"
    print(f"{name:11s} {len(r['docs']):3d} units in {r['seconds']:5.2f}s   did not read: {not_read}")

In [ ]:
# YOUR CALL: two names, before the reveal below.
# Which file do you think loses its column names inside a framework loader?
PREDICT_LOSES_COLUMNS = "support_sla_matrix.docx"
# And which one do you think a framework reads into a single block you cannot cite?
PREDICT_ONE_BLOCK = "refund_claims_march_2024.xlsx"

In [ ]:
# The same Word table row, three ways. Yours repeats the column name beside every value, so a
# piece cut out of the middle still says what the 15 minutes belongs to. Read the other two and
# decide whether you could answer "what is the first response time for a P2?" from them.
print("=== the Word table row ===")
for name in res:
    print(f"\n  {name}")
    print("   ", find_text(res, name, "support_sla_matrix.docx", "First response"))

print("\n\n=== the Excel claims sheet ===")
for name in res:
    sheet_units = [d for d in res[name]["docs"] if d.meta.get("source") == "refund_claims_march_2024.xlsx"]
    if not sheet_units:
        print(f"\n  {name}\n     never read it")
        continue
    print(f"\n  {name}: {len(sheet_units)} unit(s), the first one:")
    print("   ", " ".join(sheet_units[0].text.split())[:190])
print("\nThe sheet has a title banner above the real header row. Look at what that does to the"
      "\nframework's column names, and at how many units each loader made out of nine claims.")

In [ ]:
# The failure that matters, checked against all three. A loader that found nothing and a loader
# that is happy look identical on screen unless you count the characters yourself.
print("=== the scanned PDF ===")
for name in res:
    scan_units = [d for d in res[name]["docs"] if d.meta.get("source") == "support_escalation_sop_scanned.pdf"]
    chars = sum(len(d.text.strip()) for d in scan_units)
    flags = {k: v for d in scan_units for k, v in d.meta.items() if "error" in k or "warn" in k}
    print(f"  {name:11s} units {len(scan_units)}   characters {chars}   error or warning fields: {flags or 'none'}")

print("\n=== where the error fields actually are ===")
for name in res:
    where = sorted({d.meta.get("source", "?") for d in res[name]["docs"]
                    if any("error" in k or "warn" in k for k in d.meta)})
    print(f"  {name:11s} {', '.join(where) if where else 'no file carries one'}")
print("One of them has somewhere to report a failure, and put it on the file that worked.")

print("\n=== metadata keys each loader attached ===")
for name, keys in metadata_keys(res).items():
    print(f"  {name:11s} {len(keys):2d}  {', '.join(keys)}")
print("\nMore keys is not better. Ask of each one: can I filter on it, or cite with it?")

### Part 3. Two tools that read the layout for you

**What you will do.** Run the next three cells. The first runs Docling, a free open-source reader that
looks at the layout of each page before it reads the words. The second runs `ai_parse_document`,
Databricks' own reader, which is paid per page. The third asks every reader in this step, your loader
and your repair included, the same three questions: does clause 4.1 of the contract come back as one
line, does the Word table sit under its heading, and does any text come back from the scan.

**What you should see.** Docling reads all seven files and passes all three checks. It puts clause
4.1 back on one line, keeps the Word table under its heading as a real table, and reads 819 characters
from the scan with its own OCR. On Databricks the cell shows what Docling returned when the lab was
built, on a laptop: a live run there means a 1.8 GB install, and its first run took more than 30
minutes when this lab was built. Set `RUN_DOCLING = True` to try it anyway. On your own machine it
runs live once Docling is installed: the first run downloads about 500 MB of models, and after that
all seven files take under a minute.

`ai_parse_document` reads the six files it accepts in about 15 seconds. It keeps the table under its
heading and reads 811 characters from the scan. It fails the first check for a narrow reason: it
returns clause 4.1 as one element, which is the right unit, but keeps the PDF's line breaks inside
it. Cut its output by element and you never notice. Cut it at line breaks and you are back to the
loader's problem.

In the last cell, LangChain and LlamaIndex pass only the middle check. For PDFs they use the same
reader as your loader, pypdf, so clause 4.1 stays broken and nothing comes back from the scan. For
Word files they use a different reader, docx2txt. It keeps the table under its heading, where your
loader moves every table to the end of the file, but it puts each table cell on its own line, so a
value like "2 hours" loses its column name. None of the three checks can see that last cost.

**Why this step is here.** A layout tool does the repair you just wrote, and more, without you writing
it. It still guesses at the layout, so the same checks run on what it returns. It also costs something
the loaders above do not: a 1.8 GB install and about 500 MB of models before the first answer for
the free one, and a charge for every page for the paid one, $0.70 to $1.75 per 1,000 pages for pages
like these at Databricks' US East list price. On 40,000 files, that trade is the decision.

In [ ]:
# Docling: free, open source, and heavy. The first run in a session installs it into data/cache
# with its own copy of PyTorch and downloads its layout and OCR models. It then reads the seven
# files in a separate process, so nothing it brings can clash with the packages in this notebook.
import time
from layout_tools import (docling_ready, docling_install, docling_read, recorded, files, checks,
                          per_file as reader_rows, on_databricks)
from labkit import table

# YOUR CALL: True runs Docling live. On Databricks that means a 1.8 GB install, and its first run
# took more than 30 minutes when this lab was built, so there the default is the recorded result.
RUN_DOCLING = docling_ready() and not on_databricks()
docling = None
if RUN_DOCLING:
    try:
        t0 = time.perf_counter()
        installed = docling_install()
        docling = docling_read(files(), timeout=540)   # 9 minutes, then the recorded result
        print(f"Docling: installed in {installed:.0f}s (0 means it was already there), "
              f"read all seven files in {time.perf_counter() - t0 - installed:.0f}s")
    except RuntimeError as e:
        print("Docling could not run here:", str(e).strip().splitlines()[-1][:160])
if docling is None:
    docling = recorded("docling")["files"]
    print("This is what Docling returned when the lab was built, on a laptop.")
print(table(reader_rows(docling), width=12))
for question, answer in checks({n: r["text"] for n, r in docling.items()}).items():
    print(f"   {question:24s} {answer}")

In [ ]:
# ai_parse_document: Databricks' own reader, paid per page under AI Functions. One query sends the
# six files it accepts; it does not take spreadsheets. Anywhere but Databricks, this cell shows
# what it returned when the lab was built.
from layout_tools import ai_parse_read
if on_databricks():
    try:
        parsed, seconds = ai_parse_read(files())
        print(f"ai_parse_document: read {len(parsed)} files in {seconds:.0f}s")
    except Exception as e:      # not every workspace offers AI Functions
        parsed = recorded("ai_parse_document")["files"]
        print(f"ai_parse_document could not run in this workspace ({type(e).__name__}), "
              "so this is what it returned when the lab was built.")
else:
    parsed = recorded("ai_parse_document")["files"]
    print("ai_parse_document runs on Databricks only, so this is what it returned when the lab was built.")
print(table(reader_rows(parsed), width=12))
for question, answer in checks({n: r["text"] for n, r in parsed.items()}).items():
    print(f"   {question:24s} {answer}")
said = {n: r["errors"] for n, r in parsed.items() if r.get("errors")}
print("   errors it reported:", said or "none")

In [ ]:
# Every reader in this step, asked the same three questions. Your loader and your repair come
# from part 1, LangChain and LlamaIndex from part 2, the two layout tools from the cells above.
from layout_tools import texts_from_docs, CONTRACT, SLA
readers = {
    "your loader": texts_from_docs(docs),
    "your repair": {**texts_from_docs(docs), CONTRACT: fixed_pdf, SLA: fixed_sla},
    "LangChain": texts_from_docs(res["langchain"]["docs"]),
    "LlamaIndex": texts_from_docs(res["llamaindex"]["docs"]),
    "Docling": {n: r["text"] for n, r in docling.items()},
    "ai_parse_document": {n: r["text"] for n, r in parsed.items()},
}
print(table([{"reader": name, **checks(texts)} for name, texts in readers.items()], width=10))
print("\nOnly the two layout tools read the scan, and only Docling and your repair put clause 4.1")
print("back on one line. For PDFs the frameworks use the same reader as your loader. For Word files they")
print("use a different one, which keeps the table under its heading but puts each cell on its own line,")
print('so a value like "2 hours" loses its column name. Every reader guesses; the checks show where.')

### Which one would you use, and for what

Fill this in from your own run, not from memory:

| | reads all five formats | keeps the table structure | reads the scan | tells you extraction failed | what it costs you |
|---|---|---|---|---|---|
| the loaders in `src/freshcart.py` | | | | | |
| LangChain | | | | | |
| LlamaIndex | | | | | |
| Docling | | | | | |
| `ai_parse_document` | | | | | |

Then answer the question the debrief asks:

**You have forty minutes and a folder of contracts to index for a demo on Friday. Which do you reach
for, and what goes in the ticket so the person who inherits it knows what it does not do?**

A good answer is not "write your own". It is closer to: take the framework for the reading, keep
your own check for the counting, and write down the formats it cannot open without another package.
If the contracts are scanned or set in columns, put a layout tool in front, and run the same checks
on what it returns.

## Step 3. Cut the documents into pieces

**What you will do.** Run the next cell to cut the same 2024 refund policy three different ways and
print the first piece from each. Then make your own choice in the `YOUR CALL` cell: `STRATEGY` is
`"fixed"`, `"recursive"` or `"structural"`, with a size and an overlap counted in characters. The last
cell scores your choice and prints your worst piece in full.

**What you should see.** On this document `fixed` gives 4 pieces, `recursive` gives 2, and
`structural` gives 7. The score counts how many pieces start in the middle of a sentence, and how many
carry a number or a clause number in their first 40 characters. With the defaults, `recursive` gives 2
pieces of which 1 starts mid sentence and none start with a number, while `structural` gives 7 pieces,
none starting mid sentence and 6 starting with a clause.

**Why this step is here.** Where the cut lands decides what the search can find later. A piece that
starts mid sentence and carries no clause number cannot answer a question on its own, however good the
model is. This is the largest single quality decision in the pipeline, and you see its effect in Step
5.

A fourth approach, semantic chunking, cuts where the meaning shifts rather than where the
characters run out. It is worth its cost on documents with no headings, such as transcripts,
and it has little to offer a policy whose clauses are already numbered.

In [ ]:
# Three ways to cut one document into pieces:
#   fixed       every N characters, ignoring where sentences end
#   recursive   split on paragraphs first, then sentences, then characters
#   structural  split on the document's own headings and numbered clauses
from freshcart import chunk_fixed, chunk_recursive, chunk_structural
policy = next(d for d in docs if "refund_policy_v3_2024" in d.meta.get("source", ""))
for name, fn in (("fixed", chunk_fixed), ("recursive", chunk_recursive), ("structural", chunk_structural)):
    pieces = fn(policy.text)
    print("=" * 90)
    print(f"{name}: {len(pieces)} chunks")
    for i, p in enumerate(pieces, 1):
        print(f"  {i}. ({len(p):4d} chars) {' '.join(p.split())[:64]}")

In [ ]:
# YOUR CALL: strategy is one of "fixed", "recursive", "structural"
STRATEGY = "recursive"
SIZE = 900
OVERLAP = 120

In [ ]:
import textwrap
from freshcart import STRATEGIES
import inspect

# Two cheap signals for whether the cuts landed in sensible places:
#   starts_mid  pieces beginning with a lower-case letter, which usually means the cut
#               landed in the middle of a sentence
#   has_number  pieces carrying a digit in their first 40 characters, which here means
#               the piece kept its clause number, such as 3.1
def quality(pieces):
    starts_mid = sum(1 for p in pieces if p[:1].islower())
    has_number = sum(1 for p in pieces if any(ch.isdigit() for ch in p[:40]))
    return starts_mid, has_number

fn = STRATEGIES[STRATEGY]
kwargs = {}
params = inspect.signature(fn).parameters
if "size" in params: kwargs["size"] = SIZE
if "overlap" in params: kwargs["overlap"] = OVERLAP
mine = fn(policy.text, **kwargs)
starts_mid, has_number = quality(mine)
print(f"your setting: {STRATEGY} size={SIZE} overlap={OVERLAP}")
print(f"  {len(mine)} chunks, {starts_mid} start mid-sentence, {has_number} carry a number or clause in the first 40 characters")
print("\nfor comparison, the defaults:")
for name, f2 in STRATEGIES.items():
    p2 = f2(policy.text)
    sm, hn = quality(p2)
    print(f"  {name:11s} {len(p2):3d} chunks, {sm} mid-sentence, {hn} with a number early")
broken = [p for p in mine if p[:1].islower()]
worst = broken[0] if broken else min(mine, key=len)
label = "starts mid-sentence" if broken else "shortest"
print(f"\nyour worst chunk ({label}), verbatim. Would it answer anything on its own?")
print(textwrap.fill(" ".join(worst.split()), 90))

## Step 4. Attach labels to every piece

**What you will do.** Run the next cell to cut all seven files and print the labels carried by one
piece. Then list in the `YOUR CALL` cell the labels you would refuse to index without. The last cell
counts how many of the 30 pieces actually carry each one.

**What you should see.** 30 pieces in total. The labels on a piece are the file name, the page, the
format, the way it was cut, the document type, the effective date, and whether the document has been
replaced. With the default list, `source`, `effective_date` and `superseded` are on 100 percent of
pieces, and `page` is on only 16 of 30, which is 53 percent.

**Why this step is here.** A label that is missing cannot be filtered on later, and it cannot be shown
as a citation either. The gap in `page` is real: Word and Excel pieces have no page number, so a
citation from those files can point at the file but not at the place inside it. The `superseded` label
is the one that rescues you in Step 6.

In [ ]:
# Cut every document, not just the policy, and attach labels to each piece.
# From here on the lab works with chunks, never with whole files.
from freshcart import chunk_docs
chunks = chunk_docs(docs)
print("total chunks:", len(chunks))
print("\none chunk's metadata:")
for k, v in chunks[0].meta.items():
    print(f"   {k:16s} {v}")

In [ ]:
# YOUR CALL: the fields you would refuse to index without
REQUIRED_FIELDS = ["source", "effective_date", "superseded", "page"]

In [ ]:
# For each field you said you would insist on, count how many pieces actually carry it.
for field in REQUIRED_FIELDS:
    have = sum(1 for c in chunks if c.meta.get(field) is not None)
    print(f"   {field:16s} present on {have:3d} of {len(chunks)} chunks ({100*have/len(chunks):.0f} percent)")
print("\nA field that is missing here cannot be filtered on later, and it cannot be cited either.")
print("Look at the coverage of 'page': Word and Excel chunks have no page number, so a citation")
print("from those files can only point at the file. Decide now whether that is good enough.")

## Step 5. Index the pieces and ask five real questions

**What you will do.** Note first that the corpus holds a refund policy from 2022 and one from 2024,
with almost the same wording and different numbers. In the `YOUR CALL` cell name the file you think
comes back first for the question "how many refund claims can a customer make in thirty days", and how
sure you are. The next cell builds the index and runs five questions through it.

**What you should see.** A line saying 30 pieces are indexed, then five questions with their top three
results and a score for each. For the first question the withdrawn 2022 policy comes back first at
0.942, above the current 2024 policy at 0.795, marked `<-- SUPERSEDED`. Nothing is broken: the two
documents really are worded almost identically.

**Why this step is here.** This is the moment the whole course is built around. Similarity is working
perfectly and the answer is still wrong, because similarity has no idea which document is in force.
The fix is not a bigger model, and you apply it in the next step.

In [ ]:
# YOUR CALL
PREDICT_TOP_FILE = "refund_policy_v3_2024.pdf"
CONFIDENCE = "high"

In [ ]:
# Turn every piece into a vector and put it in a small searchable index, then ask five
# real support questions. VectorIndex().build() does the embedding and stores the matrix.
from freshcart import VectorIndex
TEST_QUERIES = [
    "how many refund claims can a customer make in thirty days",
    "who pays when food is spilled in transit",
    "how long does a customer have to submit photographic evidence",
    "when must an agent escalate to a supervisor",
    "what is the first response target for a spillage complaint",
]
idx = VectorIndex().build(chunks)
print("indexed", len(chunks), "chunks, dimension", idx.M.shape[1], "\n")
for q in TEST_QUERIES:
    print("Q:", q)
    for score, ch in idx.search(q, k=3):
        flag = "   <-- SUPERSEDED" if ch.meta.get("superseded") else ""
        print(f"   {score:.3f}  {ch.meta.get('source',''):34s} {' '.join(ch.text.split())[:60]}{flag}")
    print()

top = idx.search(TEST_QUERIES[0], k=1)[0][1].meta.get("source")
print("you predicted:", PREDICT_TOP_FILE, f"({CONFIDENCE} confidence)")
print("actually first:", top)

## Step 6. Fix it with one label, not a bigger model

**What you will do.** Run the two cells. `current_only` is a filter that keeps only the pieces whose
`superseded` label is false. Leave `APPLY_FILTER` as `True` to see the fix, or set it to `False` to
watch the damage again.

**What you should see.** The same question asked twice. Without the filter the 2022 policy leads at
0.942. With the filter only 2024 pieces come back, led by 0.795. Same model, same numbers, same
question.

**Why this step is here.** It puts a price on retrieval quality: one label, applied at search time.
Every later week adds more controls of this kind, and the habit is always the same. Fix the data and
the filter before you reach for a bigger model.

In [ ]:
# YOUR CALL: flip to False if you want to see the damage again
APPLY_FILTER = True

In [ ]:
# current_only is a one-line filter: keep a piece only if its document has not been
# withdrawn. It reads the "superseded" label you attached at chunking time.
from freshcart import current_only
q = TEST_QUERIES[0]
print("Q:", q, "\n")
print("unfiltered:")
for score, ch in idx.search(q, k=3):
    flag = "   <-- SUPERSEDED" if ch.meta.get("superseded") else ""
    print(f"   {score:.3f}  {ch.meta.get('source',''):34s}{flag}")
if APPLY_FILTER:
    print("\ncurrent documents only:")
    for score, ch in idx.search(q, k=3, where=current_only):
        print(f"   {score:.3f}  {ch.meta.get('source',''):34s}")
    print("\nSame model, same embeddings, same question. One metadata field decided the answer.")

## Step 7. Turn the pieces into vectors, three ways

**What you will do.** Run the next cell to put the same 30 pieces through three different embedders
and ask the same five questions of each. An embedder is the thing that turns a piece of text into a
list of numbers, so that "how similar are these two pieces" becomes a sum a computer can do. Then
choose one in the `YOUR CALL` cell. The last cell scores your choice, then changes the corpus
underneath it.

**What you should see.** Three sets of numbers and one uncomfortable pattern.

- `tfidf` gives 29 dimensions and puts the withdrawn 2022 policy first on the claims question, 0.942
  against 0.795 for the current one.
- `hashing` gives 256 dimensions and scores 0.372 on the same question. More dimensions is not better.
- `bge` gives 384 dimensions and is a current open-source retrieval model, the kind you would
  actually ship. It puts the withdrawn policy first as well, at 0.833 against 0.832.

Read that last pair again. One thousandth of a point separates the withdrawn policy from the one in
force. The model does not have a weak opinion about which is right. It has no opinion at all, and it
cannot have one, because the two documents say nearly the same words.

**Why this step is here.** Step 6 told you the fix is not a bigger model. This is the proof, and the
third row is what makes it stick: a real retrieval model gets it wrong on three of the five
questions, one more than the crude one. The label is still the only thing that fixed it. The last
cell shows the other half of the choice: withdraw one file from the corpus and every `tfidf` vector
moves, including vectors for pieces whose own text never changed, while `hashing` and `bge` stay
exactly where they were. Lab 9 is built on that difference.

In [ ]:
# Three ways to turn a piece of text into numbers:
#   tfidf     counts the words, then squeezes the counts into fewer dimensions.
#             It has to be fitted on the whole corpus first.
#   hashing   hashes the words into fixed slots, then projects them down.
#             It needs no fitting, so a piece's vector depends only on its own text.
#   bge       a trained open-source retrieval model, BGE small. No key and no account.
#             The model downloads once per session, about 130 MB, so the first run of
#             this cell takes longer than the rest of the notebook put together.
from freshcart import TfidfSvdEmbedder, HashingEmbedder, FastEmbedEmbedder
TEST_QUERIES = [
    "how many refund claims can a customer make in thirty days",
    "who pays when food is spilled in transit",
    "how long does a customer have to submit photographic evidence",
    "when must an agent escalate to a supervisor",
    "what is the first response target for a spillage complaint",
]
EMBEDDERS = {"tfidf": TfidfSvdEmbedder, "hashing": HashingEmbedder,
             "bge": FastEmbedEmbedder}
for key, cls in EMBEDDERS.items():
    emb = cls()
    probe = VectorIndex(emb).build(chunks)
    print("=" * 90)
    print(f"{key}: {emb.name}, {probe.M.shape[1]} dimensions")
    for q in TEST_QUERIES:
        score, ch = probe.search(q, k=1)[0]
        flag = "   <-- SUPERSEDED" if ch.meta.get("superseded") else ""
        print(f"   {score:.3f}  {ch.meta.get('source',''):34s} {q[:44]}{flag}")

In [ ]:
# YOUR CALL: which embedder would you put in the pipeline?
EMBEDDER = "tfidf"

In [ ]:
# Your choice, scored on the same five questions, then tested for something else
# entirely: what happens to a stored vector when a DIFFERENT document changes.
import numpy as np
emb = EMBEDDERS[EMBEDDER]()
my_index = VectorIndex(emb).build(chunks)
stale = sum(1 for q in TEST_QUERIES if my_index.search(q, k=1)[0][1].meta.get("superseded"))
print(f"your choice: {EMBEDDER} ({emb.name}), {my_index.M.shape[1]} dimensions")
print(f"   the top hit is a withdrawn document on {stale} of the {len(TEST_QUERIES)} questions\n")

# Withdraw one file from the corpus. Nothing else changes, and the piece we watch
# comes from a different file whose text is untouched.
texts = [c.text for c in chunks]
smaller = [c.text for c in chunks if c.meta.get("source") != "support_qbr_q1_2024.pptx"]
watch = texts[0]
print(f"one file is withdrawn: {len(texts)} pieces become {len(smaller)}, and we watch a piece")
print("from partner_agreement_extract.pdf, whose own text did not change:")
for key, cls in EMBEDDERS.items():
    before = cls().fit(texts).encode([watch])[0]
    after = cls().fit(smaller).encode([watch])[0]
    # Compare the whole vector, not part of it: after a refit, position 1 of a tfidf vector
    # measures something different from position 1 before, so a partial comparison means nothing.
    if len(before) != len(after):
        verdict = "a different length, so the stored vector no longer fits the index"
    elif np.allclose(before, after):
        verdict = "identical, so the stored vector can be reused"
    else:
        verdict = "different numbers, so the stored vector no longer matches"
    print(f"   {key:8s} {len(before):3d} -> {len(after):3d} numbers: {verdict}")
print("\nA vector that moves when OTHER documents change cannot be cached, so every")
print("nightly load re-embeds the whole corpus. Lab 9 is where that bill arrives.")

## Step 8. Write down what you found

**What you will do.** Run the cell. It prints the measured lines from your own run and lists the three
sentences you have to write yourself. Paste the whole block into the **Your write-up** cell at the bottom of this notebook.

**What you should see.** Six measured lines: how many files loaded, what extracted nothing, your
quarantine rule, your chunking choice, the total number of pieces, and the file that came first for
the claims question. Then three questions no script can answer for you.

**Why this step is here.** These numbers are the starting point for Week 2, where you fix these same
failures with better retrieval and measure whether you actually improved anything.

In [ ]:
# Collect the numbers you produced in this notebook, ready to paste into your answer sheet.
measured = [
    f"- files loaded: {len(docs)} document units from {len(set(d.meta.get('source') for d in docs))} files",
    f"- extracted nothing: {', '.join(empty) or 'none'}",
    f"- your quarantine rule: fewer than {MIN_CHARS_PER_UNIT} characters per unit",
    f"- your chunking: {STRATEGY}, size {SIZE}, overlap {OVERLAP}, {len(mine)} chunks on the 2024 policy",
    f"- total chunks indexed: {len(chunks)}",
    f"- first result for the claims question, unfiltered: {top}",
]
print("\n".join(measured))
print("\nNow scroll to the Your write-up cell and finish these three, in your own words:")
print("  1. The worst chunk you produced, pasted verbatim, and why it is useless alone.")
print("  2. One query that should have worked and did not. What did it return instead?")
print("  3. Which format lost the most information, and what you would do about it in production.")

## When you are done

- In your index the scanned file is still empty, and it stays that way for the rest of Lab 1. The two
  tools in Step 2 read it, but nothing after Step 2 uses what they returned. Putting it in your index
  is assignment A1, which you get at the end of Day 2.
- The Azure version runs the same steps against Azure Document Intelligence, Azure OpenAI embeddings
  and Azure AI Search: `python labs\lab01_azure.py --azure`.
- Keep the questions that failed. They become the test set in Week 2.

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it, and that export is what you
hand in.

Do not write in `handouts/lab1_answers.md`. That folder is rebuilt from the lab pack every session, so anything you
type there is gone tomorrow.

---

**Pair:**

### 1. Which source extracted zero characters, and why?

### 2. What would you do about it in production?

### 3. Your chunking strategy per document type, and why

| Document type | Strategy | Reason |
|---|---|---|
| Policy PDFs | | |
| Slide deck | | |
| Spreadsheet | | |
| SLA table (DOCX) | | |

### 4. The worst chunk you produced: paste it verbatim

### 5. The query where the superseded policy outranked the current one

Query:

Why did similarity not help?

What fixed it?

### 6. One query that should have worked and did not. What was missing?

### 7. Step 2 repaired two files that loaded without an error. Which repair would you ship first, and what check would tell you it had stopped working?